In [1]:
# Cell 1: 依存関係の確認とインポート
import sys
from pathlib import Path

# ifcopenshell, networkx, rdflib の確認
import importlib
def ensure_package(pkg):
    try:
        importlib.import_module(pkg)
        return True
    except ImportError:
        return False

missing = [p for p in ["ifcopenshell", "networkx", "rdflib"] if not ensure_package(p)]
if missing:
    print(f"Installing: {missing}")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import ifcopenshell
import networkx as nx
from rdflib import Graph, Namespace, RDF, RDFS, Literal

print("✓ All dependencies loaded")

✓ All dependencies loaded


In [2]:
# Cell 2: プロジェクトパスと設定
project_root = Path(r"C:\Users\Owner\Documents\25952501_RioSaijo\IFC_ToBrick_IoT_Interoperability_Project")
sys.path.insert(0, str(project_root / "src"))

IFC_PATH = project_root / "data" / "In" / "ifc" / "curated" / "51M-SKA-00-00-M3-MEP-0001_IFC 2x3-P01_with_port_relations.ifc"
TTL_OUT = project_root / "data" / "Out" / "equipment_topology.ttl"

Path(TTL_OUT).parent.mkdir(parents=True, exist_ok=True)

print(f"IFC Input: {IFC_PATH}")
print(f"TTL Output: {TTL_OUT}")
assert IFC_PATH.exists(), f"IFC not found: {IFC_PATH}"

IFC Input: C:\Users\Owner\Documents\25952501_RioSaijo\IFC_ToBrick_IoT_Interoperability_Project\data\In\ifc\curated\51M-SKA-00-00-M3-MEP-0001_IFC 2x3-P01_with_port_relations.ifc
TTL Output: C:\Users\Owner\Documents\25952501_RioSaijo\IFC_ToBrick_IoT_Interoperability_Project\data\Out\equipment_topology.ttl


In [3]:
# Cell 3: モジュールインポート
from equip_topology_reasoner.equip_reasoner import (
    load_ifc,
    build_equip_graph,
    filter_graph_by_bdns_csv,
    export_brick_ttl,
    load_bdns_guids,
)

print("✓ equip_reasoner functions imported")

✓ equip_reasoner functions imported


In [4]:
# Cell 4: IFCロードとグラフ構築
model = load_ifc(str(IFC_PATH))
print(f"✓ IFC loaded")

# 指定ID範囲の要素のみを処理（#2676166から#2676344）
G = build_equip_graph(model, start_id=2676166, end_id=2676344)
print(f"✓ Graph built: {len(G.nodes())} nodes, {len(G.edges())} edges")

✓ IFC loaded
✓ Graph built: 36 nodes, 35 edges


In [5]:
# Cell 5: BDNSフィルタ設定と適用（CSVベース）
BDNS_CSV_PATH = project_root / "data" / "intermediate" / "brick_equipment_set.csv"

# CSVのGUIDリストでフィルタリング
G_bdns = filter_graph_by_bdns_csv(G, str(BDNS_CSV_PATH))
print(f"✓ BDNS filter applied: {len(G_bdns.nodes())} nodes, {len(G_bdns.edges())} edges")

✓ BDNS filter applied: 7 nodes, 1 edges


In [6]:
# Cell 6: Brick TTL出力と検証
# BDNS CSVからBrickクラスマッピングを使用
export_brick_ttl(model, G_bdns, str(TTL_OUT), str(BDNS_CSV_PATH), include_ports=True)
print(f"✓ TTL exported to: {TTL_OUT}")

# TTLファイルの検証
ttl_graph = Graph()
ttl_graph.parse(str(TTL_OUT), format="turtle")

BRICK = Namespace("https://brickschema.org/schema/Brick#")
EX = Namespace("https://example.org/building#")

feeds_triples = list(ttl_graph.triples((None, BRICK.feeds, None)))
hasPart_triples = list(ttl_graph.triples((None, BRICK.hasPart, None)))
port_triples = list(ttl_graph.triples((None, RDF.type, BRICK.Port)))

print(f"\n📊 TTL Statistics:")
print(f"  Total triples: {len(ttl_graph)}")
print(f"  feeds relationships: {len(feeds_triples)}")
print(f"  hasPart relationships: {len(hasPart_triples)}")
print(f"  Port instances: {len(port_triples)}")
print(f"  File size: {Path(TTL_OUT).stat().st_size} bytes")

# 接続サンプル表示
if feeds_triples:
    print(f"\n✓ Sample feeds relationships (first 10):")
    for i, (subj, pred, obj) in enumerate(feeds_triples[:10]):
        subj_name = str(subj).split("#")[-1]
        obj_name = str(obj).split("#")[-1]
        print(f"  {i+1}. {subj_name} → {obj_name}")

https://brickschema.org/schema/Brick#Smoke_Damper | Fire_Damper does not look like a valid URI, trying to serialize this will break.
https://brickschema.org/schema/Brick#Smoke_Damper | Fire_Damper does not look like a valid URI, trying to serialize this will break.
https://brickschema.org/schema/Brick#Smoke_Damper | Fire_Damper does not look like a valid URI, trying to serialize this will break.
https://brickschema.org/schema/Brick#Smoke_Damper | Fire_Damper does not look like a valid URI, trying to serialize this will break.
https://brickschema.org/schema/Brick#Smoke_Damper | Fire_Damper does not look like a valid URI, trying to serialize this will break.
https://brickschema.org/schema/Brick#Smoke_Damper | Fire_Damper does not look like a valid URI, trying to serialize this will break.
https://brickschema.org/schema/Brick#Smoke_Damper | Fire_Damper does not look like a valid URI, trying to serialize this will break.
https://brickschema.org/schema/Brick#Smoke_Damper | Fire_Damper does 

✓ TTL exported to: C:\Users\Owner\Documents\25952501_RioSaijo\IFC_ToBrick_IoT_Interoperability_Project\data\Out\equipment_topology.ttl

📊 TTL Statistics:
  Total triples: 16
  feeds relationships: 1
  hasPart relationships: 0
  Port instances: 0
  File size: 677 bytes

✓ Sample feeds relationships (first 10):
  1. DUW-00403 → SG-00252
